# Word Tokenization and Vocabulary Construction

Implements basic word-level tokenization, vocabulary mapping, and out-of-vocabulary handling from scratch.


Loads the raw text corpus used to construct the vocabulary.


In [ ]:
with open("/home/lakshay/BuildingLargeLanguageModel/src/buildinglargelanguagemodel/data/the-verdict.txt", "r") as file:
    content = file.read()
    print(content)

Splits text on punctuation and whitespace into individual tokens using regular expressions.


In [76]:
import re

text = "Hello @world! This costs $50."
result = re.split(r'(\s)', text)

print(result)

['Hello', ' ', '@world!', ' ', 'This', ' ', 'costs', ' ', '$50.']


Tokenizes the full corpus using regular expression splitting.


In [77]:
import re

preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', content)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])


['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


Displays the total number of extracted tokens.


In [78]:
len(preprocessed)


4690

Extracts unique tokens from the preprocessed text and maps each token to a unique integer ID.


In [79]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)


1130


Constructs the reverse mapping from integer IDs back to string tokens.


In [ ]:
# Assign an integer to each token in the vocab
vocab = {token:integer for integer, token in enumerate(all_words)}
vocab

Implements a basic bidirectional tokenizer for encoding text and decoding token IDs.


In [81]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

Verifies the tokenizer using sample text from the training corpus.


In [82]:
tokenizer = SimpleTokenizerV1(vocab)

text = """It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""

encoded = tokenizer.encode(text)
print("encoded:", encoded)

decoded = tokenizer.decode(encoded)
print("decoded:", decoded)

encoded: [56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
decoded: It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


Adds `<|unk|>` and `<|endoftext|>` special tokens to handle unseen words and document boundaries.


In [83]:
all_words = sorted(list(set(preprocessed)))
all_words.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_words)}


Displays the updated vocabulary size and special token assignments.


In [84]:
len(vocab.items())

1132

In [85]:
for item in list(vocab.items())[-5:]:
    print(item)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


Implements `SimpleTokenizerV2` with fallback handling for out-of-vocabulary tokens.


In [86]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

Tests `SimpleTokenizerV2` on concatenated documents and an out-of-vocabulary word.


In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))

encoded = tokenizer.encode(text)
print("encoded:", encoded)

decoded = tokenizer.decode(encoded)
print("decoded:", decoded)

unk_word = "ArithmeticError"

encoded_unk = tokenizer.encode(unk_word)
print("encoded unk:", encoded_unk)

decoded_unk = tokenizer.decode(encoded_unk)
print("decoded unk:", decoded_unk)


Original text: Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
encoded: [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
decoded: <|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
encoded unk: [1131]
decoded unk: <|unk|>
